# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulm111/ML-Assignement01/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Ranking, not point-accuracy ** `training-honest-models`'s
method table says a "which first?" ranking question calls for a classifier's probability evaluated at
precision@K. Lane 4 is exactly that kind of question. But precision@K needs an observed label  "was
flagging this page actually right"  and there isn't one: no future outcome to check against, and
deriving a label from `ctr_gap` itself would just be scoring the rule against a copy of itself. w04
already named this as an open gap and left it undisguised rather than fake it; that deferral still
holds. So w02's locked framing (target = observed `ctr`, regression) is the honest path available
here, not a shortcut around the skill's table  and because a regression is standing in for a ranking
decision, Section 3 adds two label-free ranking checks (top-20 queue overlap, rank correlation)
alongside the accuracy metrics, so a model can't look good on R² while quietly reshuffling the review
queue for no reason.

Logistic Regression is out for the same reason precision@K is: no legitimate binary label to fit
against. Clustering is out because the lane needs one comparable page-level score, not a group name.

**What the model gets that the Week-4 rule never had:** the rule only ever saw `position_tier` (5
hand-cut buckets) and `content_type`. This model adds everything else from the w03 contract the rule
left on the table — raw `avg_position_month`, `impressions_month`, and the GA4/AI engagement features
(`ga4_engaged_sessions_month`, `sessions_ai_month`, `scroll_events_month`), with a `has_ga4_month` flag
so a month with no GA4 rows isn't silently read as zero engagement.

**Method, in order:** shallow **Decision Tree Regressor** (depth-4, printable) first, then **Random
Forest**. Section 3 picks between them with an explicit rule, not "whichever number is bigger"  if
the forest doesn't clear the tree by a real margin, the tree wins, on the grounds that a model you can
print and read is worth more than a black box a fraction of a point stronger. Gradient Boosting stays
off the table entirely unless that check shows real signal left behind  a marginal RF-over-tree gap
already answers whether boosting is worth its extra validation care, so it isn't run blind.

**If nothing beats the frozen Week-4 rule, that's a legitimate finding**, not a failed assignment  it
would mean two buckets already capture most of what's there.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd, os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

con.execute(f"""
    CREATE OR REPLACE TABLE march_facts AS
    SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""")

# GSC side: identical monthly aggregation to the Week-4 baseline, so the two
# are comparable on the same rows.
gsc_monthly = con.sql("""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_month
    FROM march_facts
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
""").df()

# GA4 side: sum ONLY days where ga4_data_available IS TRUE. Rows before a
# client's ga4_data_start are zero-filled with the flag FALSE  summing
# those in would quietly turn "no data" into "zero engagement".
ga4_monthly = con.sql("""
    SELECT
        client_hash_id, content_hash_id,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_month,
        SUM(sessions_ai) AS sessions_ai_month,
        SUM(scroll_events) AS scroll_events_month,
        COUNT(*) AS ga4_days_available
    FROM march_facts
    WHERE ga4_data_available IS TRUE
    GROUP BY 1, 2
""").df()

monthly = gsc_monthly.merge(ga4_monthly, on=["client_hash_id", "content_hash_id"], how="left")

# has_ga4_month flag instead of a blind fillna(0)  keeps "no GA4 rows this
# month" distinguishable from "GA4 rows present, engagement genuinely zero".
monthly["has_ga4_month"] = monthly["ga4_days_available"].notna().astype(int)
for c in ["ga4_engaged_sessions_month", "sessions_ai_month", "scroll_events_month"]:
    monthly[c] = monthly[c].fillna(0)
monthly = monthly.drop(columns=["ga4_days_available"])

monthly["position_tier"] = np.select(
    [monthly["avg_position_month"] <= 3, monthly["avg_position_month"] <= 10,
     monthly["avg_position_month"] <= 20, monthly["avg_position_month"] <= 50],
    ["top_3", "page_1", "striking", "page_3_5"], default="deep"
)

content = con.sql(f"""
    SELECT content_hash_id, content_type, is_published, is_deleted
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()

q = monthly.merge(content, on="content_hash_id", how="inner")
q = q[(q["is_published"] == True) & (q["is_deleted"] == False)]
q = q[(q["impressions_month"] >= 100) & (q["avg_position_month"] >= 1)].copy()
q["ctr"] = q["clicks_month"] / q["impressions_month"]

print(f"Rows in modeling set (same filters as the Week-4 baseline): {len(q)}")
print(f"Distinct clients: {q['client_hash_id'].nunique()}")
print(f"has_ga4_month rate: {q['has_ga4_month'].mean():.1%}")
q.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in modeling set (same filters as the Week-4 baseline): 100702
Distinct clients: 44
has_ga4_month rate: 55.3%


,client_hash_id,content_hash_id,impressions_month,clicks_month,avg_position_month,ga4_engaged_sessions_month,sessions_ai_month,scroll_events_month,has_ga4_month,position_tier,content_type,is_published,is_deleted,ctr
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.825362,0.0,0.0,0.0,0,page_1,keyword article,True,False,0.001112
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,0.0,5.136778,0.0,0.0,0.0,0,page_1,keyword article,True,False,0.000000
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,1.0,4.818653,0.0,0.0,0.0,0,page_1,keyword article,True,False,0.001295
5,client_62f4a7e64f5e0096,content_225dc9235023be5f,488.0,1.0,18.657787,0.0,0.0,0.0,0,striking,keyword article,True,False,0.002049
6,client_62f4a7e64f5e0096,content_babcf791dccc1610,181.0,0.0,11.740331,0.0,0.0,0.0,0,striking,keyword article,True,False,0.000000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client, not by row.** Rows for the same client share hidden character  brand strength,
template quality, how aggressively they publish — that a random row-level split would let leak across
train and test (the model could partly learn "this is client X's page" through the client's other pages
in train, even without the hash ID as a feature). `client_hash_id` is safe to group/split on per the
data contract (pseudonym, never a feature).

**Not time-aware.** Single-month cross-section (`2026-03`) predicting a same-month outcome  no "later"
to hold out yet. A time split is the right call once a real future-outcome label exists.

**Split:** 80/20 by unique `client_hash_id`, fixed seed, verified to have zero client overlap  **then
checked against a naive random split**, per `hunting-leakage-and-validating`'s own verify step: *"swap
your random split for a grouped split and report both numbers."* A quick Random Forest is fit both ways
below; if grouping didn't matter the two numbers land close, if it did the random split looks
artificially better  that gap is client-level memorization, not modeling skill, and it's reported
either way rather than assumed.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(q, groups=q["client_hash_id"]))
train, test = q.iloc[train_idx].copy(), q.iloc[test_idx].copy()

overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"Train rows: {len(train)}  ({train['client_hash_id'].nunique()} clients)")
print(f"Test rows:  {len(test)}  ({test['client_hash_id'].nunique()} clients)")
print(f"Clients present in both train and test (must be 0): {len(overlap)}")

Train rows: 93693  (35 clients)
Test rows:  7009  (9 clients)
Clients present in both train and test (must be 0): 0


In [16]:
# Honest-validation sanity check: does the grouped split actually change anything,
# or would a naive random split have told the same story? Same features, same
# model class, same seed  only the split changes.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

CHECK_COLS = ["avg_position_month", "impressions_month", "ga4_engaged_sessions_month",
              "sessions_ai_month", "scroll_events_month", "has_ga4_month"]

def make_X(df, ref_cols=None):
    X = pd.get_dummies(df[CHECK_COLS + ["content_type"]], columns=["content_type"])
    return X.reindex(columns=ref_cols, fill_value=0) if ref_cols is not None else X

rf_check_kwargs = dict(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)

# naive random row-level split
r_train, r_test = train_test_split(q, test_size=0.2, random_state=42)
r_train_X = make_X(r_train)
r_test_X = make_X(r_test, r_train_X.columns)
rf_random = RandomForestRegressor(**rf_check_kwargs).fit(r_train_X, r_train["ctr"])
random_mse = mean_squared_error(r_test["ctr"], rf_random.predict(r_test_X))

# grouped split (the one this notebook actually uses)
g_train_X = make_X(train)
g_test_X = make_X(test, g_train_X.columns)
rf_grouped = RandomForestRegressor(**rf_check_kwargs).fit(g_train_X, train["ctr"])
grouped_mse = mean_squared_error(test["ctr"], rf_grouped.predict(g_test_X))

print(f"Random Forest test MSE -- naive random split:   {random_mse:.6f}")
print(f"Random Forest test MSE -- grouped by client:     {grouped_mse:.6f}")
print(f"Gap: {(1 - random_mse / grouped_mse) * 100:+.1f}% "
      "(positive = random split looked artificially better -- client leakage was inflating it)")

Random Forest test MSE -- naive random split:   0.000014
Random Forest test MSE -- grouped by client:     0.000031
Gap: +55.5% (positive = random split looked artificially better -- client leakage was inflating it)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Metric: same one w02 committed to**  % reduction in CTR variance (MSE) vs. the tier-only baseline —
reported alongside plain R² and MAE (CTR points), plus the base rate (test-set mean CTR), all on the
same held-out test rows for every method.

**The Week-4 rule baseline is frozen, not refit.** The first draft of this notebook refit it on
train-only, which quietly broke `building-baselines`'s own rule: *"keep the baseline frozen once the
model work starts."* Fixed here  the rule is recomputed with the exact same code and the same full
March panel as w04 (so it's reproducible in this notebook run, per `training-honest-models`'s verify
step), then simply *applied* to the held-out test rows. Its group averages technically include the test
rows themselves  same as w04's did. That's what a frozen rule baseline is; it doesn't get to move, and
the models below still only ever see train rows, which is the harder bar, not a softer one.

**Below, in order:** baseline applied → models trained → one comparison table → an explicit
"does not reward complexity alone" decision between the tree and the forest → the two ranking
diagnostics from Section 1.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Frozen exactly as in w04 -- computed on the FULL March panel, NOT refit on
# train. Per building-baselines: "keep the baseline frozen once the model
# work starts." Recomputed here (not reloaded from a gitignored CSV) so it's
# reproducible in this run, per training-honest-models' verify step.
MIN_STRATUM_N = 30

tier_mean_full = q.groupby("position_tier")["ctr"].mean()
global_mean_full = q["ctr"].mean()
strat_full = q.groupby(["position_tier", "content_type"])["ctr"].agg(["mean", "count"])

def tier_only_predict(df):
    return df["position_tier"].map(tier_mean_full).fillna(global_mean_full).values

def week4_rule_predict(df):
    preds = np.empty(len(df))
    for i, (tier, ctype) in enumerate(zip(df["position_tier"], df["content_type"])):
        key = (tier, ctype)
        if key in strat_full.index and strat_full.loc[key, "count"] >= MIN_STRATUM_N:
            preds[i] = strat_full.loc[key, "mean"]
        else:
            preds[i] = tier_mean_full.get(tier, global_mean_full)
    return preds

test["pred_tier_only"] = tier_only_predict(test)
test["pred_week4_rule"] = week4_rule_predict(test)

print("Baselines frozen exactly as in w04 (full March panel), applied to held-out test rows.")

Baselines frozen exactly as in w04 (full March panel), applied to held-out test rows.


In [18]:
import sklearn
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

print("scikit-learn version:", sklearn.__version__)  # tree-ensemble numbers can shift a bit across versions

FEATURE_COLS = ["avg_position_month", "impressions_month",
                 "ga4_engaged_sessions_month", "sessions_ai_month",
                 "scroll_events_month", "has_ga4_month"]

train_X = pd.get_dummies(train[FEATURE_COLS + ["content_type"]], columns=["content_type"])
test_X = pd.get_dummies(test[FEATURE_COLS + ["content_type"]], columns=["content_type"])
test_X = test_X.reindex(columns=train_X.columns, fill_value=0)  # align dummy columns

y_train, y_test = train["ctr"], test["ctr"]

tree = DecisionTreeRegressor(max_depth=4, min_samples_leaf=50, random_state=42)
tree.fit(train_X, y_train)
test["pred_tree"] = tree.predict(test_X)

forest = RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                random_state=42, n_jobs=-1)
forest.fit(train_X, y_train)
test["pred_forest"] = forest.predict(test_X)

print(f"Trained on {len(train_X)} rows, {train_X.shape[1]} features: {list(train_X.columns)}")

scikit-learn version: 1.6.1
Trained on 93693 rows, 9 features: ['avg_position_month', 'impressions_month', 'ga4_engaged_sessions_month', 'sessions_ai_month', 'scroll_events_month', 'has_ga4_month', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article']


In [19]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

tier_mse = mean_squared_error(y_test, test["pred_tier_only"])

rows = []
for label, col in [
    ("Tier-only baseline (Week-2 reference)", "pred_tier_only"),
    ("Week-4 rule (tier x content_type, frozen)", "pred_week4_rule"),
    ("Decision Tree (depth 4)", "pred_tree"),
    ("Random Forest (300 trees)", "pred_forest"),
]:
    pred = test[col]
    mse = mean_squared_error(y_test, pred)
    rows.append({
        "model": label,
        "r2_vs_test_mean": round(r2_score(y_test, pred), 4),
        "mae_ctr_pts": round(mean_absolute_error(y_test, pred), 4),
        "pct_variance_reduction_vs_tier_baseline": round((1 - mse / tier_mse) * 100, 2),
    })

comparison = pd.DataFrame(rows)
comparison["base_rate_mean_ctr"] = round(y_test.mean(), 4)
comparison

,model,r2_vs_test_mean,mae_ctr_pts,pct_variance_reduction_vs_tier_baseline,base_rate_mean_ctr
0,Tier-only baseline (Week-2 reference),-0.0070,0.0036,0.00,0.004
1,"Week-4 rule (tier x content_type, frozen)",-0.0083,0.0036,-0.13,0.004
2,Decision Tree (depth 4),0.1035,0.0033,10.97,0.004
3,Random Forest (300 trees),0.1452,0.0032,15.11,0.004


In [20]:
# "Does not reward complexity alone"  an explicit rule, not eyeballing the table.
COMPLEXITY_MARGIN = 1.0  # percentage points of variance reduction the forest must clear

tree_row = comparison.loc[comparison["model"] == "Decision Tree (depth 4)"].iloc[0]
forest_row = comparison.loc[comparison["model"] == "Random Forest (300 trees)"].iloc[0]
gap = forest_row["pct_variance_reduction_vs_tier_baseline"] - tree_row["pct_variance_reduction_vs_tier_baseline"]

if gap >= COMPLEXITY_MARGIN:
    best_model_label, best_col = "Random Forest (300 trees)", "pred_forest"
    reason = f"Random Forest beats the tree by {gap:.2f} pts of variance reduction -- worth the extra complexity."
else:
    best_model_label, best_col = "Decision Tree (depth 4)", "pred_tree"
    reason = f"Random Forest only beats the tree by {gap:.2f} pts -- not enough to give up a model you can print and read. Keeping the tree."

print(f"Chosen model: {best_model_label}")
print(reason)

Chosen model: Random Forest (300 trees)
Random Forest beats the tree by 4.14 pts of variance reduction -- worth the extra complexity.


In [21]:
# The lane's real decision is a RANKING ("which pages first"), not a point CTR
# guess. Per training-honest-models, "which first?" problems call for
# precision@K  but that needs an observed label for "was reviewing this page
# actually right", and there isn't one (see Section 1). So: two label-free
# ranking diagnostics on the same test rows instead of a faked precision@K.
from scipy.stats import spearmanr

QUEUE_FILTER = test["position_tier"] != "deep"  # matches w04's queue eligibility

def opportunity_rank(pred_col):
    gap = test.loc[QUEUE_FILTER, pred_col] - test.loc[QUEUE_FILTER, "ctr"]
    score = gap * test.loc[QUEUE_FILTER, "impressions_month"]
    return score.sort_values(ascending=False)

rule_rank = opportunity_rank("pred_week4_rule")
model_rank = opportunity_rank(best_col)

top20_overlap = len(set(rule_rank.head(20).index) & set(model_rank.head(20).index))
rho, _ = spearmanr(rule_rank.loc[model_rank.index], model_rank)

print(f"Top-20 review queue overlap ({best_model_label} vs. frozen Week-4 rule): {top20_overlap}/20 pages")
print(f"Spearman rank correlation across the full queue: {rho:.3f}")

Top-20 review queue overlap (Random Forest (300 trees) vs. frozen Week-4 rule): 12/20 pages
Spearman rank correlation across the full queue: 0.891


**Results, for the record.** Both baselines are functionally noise on held-out clients — tier-only
scores R² = -0.007 and the frozen Week-4 rule R² = -0.008, i.e. *worse* than just guessing the test
set's own mean CTR for every row. Decision Tree reaches R² = 0.104 (11.0% MSE-based variance
reduction), Random Forest R² = 0.146 (15.2%) — the forest clears the 1-point complexity bar (+4.18 pts
over the tree) and is the model taken forward. Read the 15.2% headline against MAE too: MAE improves
0.0036 → 0.0032, about an 11% reduction — smaller than the MSE-based number because CTR is right-skewed
and MSE weights the tail harder. Both are real; report both, not just the bigger one.

On ranking — the metric that actually matches the lane — the Random Forest's top-20 review queue
overlaps the frozen rule's by 12/20: **8 of the top 20 "review this first" pages change** depending on
which method is trusted, against a 0.892 rank correlation across the full eligible queue.

The split design mattered more than any single model result above: a naive random split understated
held-out error by roughly 2x (grouped-split MSE 0.000031 vs. random-split MSE 0.000015). Without the
grouped split, none of the numbers above would be trustworthy.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Where the model is wrong, and what it leans on.** Error is highest at `top_3` (MAE 0.0040) and
falls monotonically to `deep` (MAE 0.0012)  expected, not concerning: top-of-page CTR has real
headroom to be wrong about, while `deep`-tier CTR is compressed near zero for every page, so small
absolute misses there are mechanical, not evidence of better modeling.

Impression volume: error falls cleanly from 100-500 through 2k-10k impressions (0.0035 → 0.0028 →
0.0026) — more impressions, less noisy CTR, easier to predict, as expected — then the 10k+ bucket's
*mean* jumps to 0.0044. That's not a systematic high-volume blind spot: its *median* (0.0020) sits
right in line with every other bucket, and n=41 is thin enough that a couple of outlier misses swing
the mean on their own. Mean and median are both printed below so this isn't just asserted.

**Content_type needs a caveat, not a conclusion.** Permutation importance reads exactly 0.0 for all
three content_type dummies — but the entire held-out test fold turned out to be `keyword article` (all
7,009 rows). Not bad luck from a small split — `keyword article` is 98.0% of the full 100,702-row
panel — but the deeper issue, checked below: content_type is closer to a **client-level constant** than
a page-level signal. For the median client, 100% of their pages share one content_type. That reframes
two earlier results rather than just excusing a missing metric: it's a plausible reason the frozen
Week-4 rule's `(tier × content_type)` stratification failed to generalize to new clients (R² = -0.008)
— a "content_type" cut that's mostly tagging client identity won't transfer to clients the rule never
priced in, any more than tier alone did — and it's a plausible reason the trained models generalize
where the rule doesn't: permutation importance shows they lean on `ga4_engaged_sessions_month`,
`avg_position_month`, `has_ga4_month`, and `impressions_month` instead — genuinely page-level, portable
signals, none of them client-proxies. `ga4_engaged_sessions_month` was checked against the w03 contract
(same-day on-page behavior, a different data source from GSC clicks) — it's correlated with CTR through
shared page quality, not circularly derived from the label.

The three worst individual misses are all low-impression rows (140-259 impressions) where the model
badly underpredicts an unusually high real CTR (predicting ~0.003-0.009 against actual 0.07-0.08) —
consistent with how tree ensembles behave: predictions are averages over similar training rows, so a
genuinely exceptional low-volume page gets pulled toward its peer group's average. Worth naming as a
real limitation: this model is better at the aggregate ranking than at flagging the single rare
outperformer.

In [22]:
# Evidence for the content_type caveat above -- not just asserted, printed.
print("Content-type distribution, full panel:")
print(q["content_type"].value_counts())

print("\nContent-type distribution, test fold:")
print(test["content_type"].value_counts())

print("\nPer-client dominant content_type share (how client-specific content_type really is):")
print(q.groupby("client_hash_id")["content_type"]
      .agg(lambda s: s.value_counts(normalize=True).iloc[0])
      .describe())

Content-type distribution, full panel:
content_type
keyword article       98665
feedly article         1040
comparison article      997
Name: count, dtype: int64

Content-type distribution, test fold:
content_type
keyword article    7009
Name: count, dtype: int64

Per-client dominant content_type share (how client-specific content_type really is):
count    44.000000
mean      0.946738
std       0.117220
min       0.563265
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: content_type, dtype: float64


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
test["abs_error"] = (test["ctr"] - test[best_col]).abs()
print("Mean absolute error by position tier:")
print(test.groupby("position_tier")["abs_error"].mean().sort_values(ascending=False))

print("\nMean absolute error by content type:")
print(test.groupby("content_type")["abs_error"].mean().sort_values(ascending=False))

test["impression_bucket"] = pd.cut(
    test["impressions_month"], bins=[0, 500, 2000, 10000, np.inf],
    labels=["100-500", "500-2k", "2k-10k", "10k+"]
)
print("\nMean absolute error by impression volume (mean AND median -- mean >> median flags outlier-driven, not systematic):")
print(test.groupby("impression_bucket", observed=True)["abs_error"].agg(["mean", "median", "count"]))

Mean absolute error by position tier:
position_tier
top_3       0.004027
page_1      0.003667
striking    0.003160
page_3_5    0.002736
deep        0.001158
Name: abs_error, dtype: float64

Mean absolute error by content type:
content_type
keyword article    0.003237
Name: abs_error, dtype: float64

Mean absolute error by impression volume (mean AND median -- mean >> median flags outlier-driven, not systematic):
                       mean    median  count
impression_bucket                           
100-500            0.003520  0.001923   4231
500-2k             0.002839  0.001771   2136
2k-10k             0.002575  0.001638    601
10k+               0.004382  0.001989     41


In [24]:
from sklearn.inspection import permutation_importance

model_map = {"Decision Tree (depth 4)": tree, "Random Forest (300 trees)": forest}
best_model = model_map[best_model_label]

perm = permutation_importance(best_model, test_X, y_test, n_repeats=10,
                               random_state=42, scoring="r2")
importance = pd.Series(perm.importances_mean, index=test_X.columns).sort_values(ascending=False)
print(f"Permutation importance on held-out test ({best_model_label}):")
importance.head(10)

Permutation importance on held-out test (Random Forest (300 trees)):


,0
ga4_engaged_sessions_month,0.186260
avg_position_month,0.120632
has_ga4_month,0.060963
impressions_month,0.052377
content_type_comparison article,0.000000
content_type_feedly article,0.000000
content_type_keyword article,0.000000
sessions_ai_month,-0.000005
scroll_events_month,-0.002313


In [25]:
worst = test.sort_values("abs_error", ascending=False).head(3)[[
    "content_hash_id", "position_tier", "content_type", "avg_position_month",
    "impressions_month", "has_ga4_month", "ctr", best_col, "abs_error"
]]
worst

,content_hash_id,position_tier,content_type,avg_position_month,impressions_month,has_ga4_month,ctr,pred_forest,abs_error
54644,content_04e3473c74438c83,top_3,keyword article,2.850000,140.0,1,0.078571,0.002873,0.075698
130618,content_b96495c6b2911b12,page_3_5,keyword article,33.693750,160.0,1,0.075000,0.005176,0.069824
42062,content_0a124186f945b8cd,striking,keyword article,18.444015,259.0,1,0.073359,0.008497,0.064862


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.